# Image Captioning using VGG19 and PyTorch
This notebook implements an image captioning system. It loads pre-extracted VGG19 features and trains an Encoder-Decoder LSTM to generate captions for images.

In [1]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.notebook import tqdm
import os

from src.dataset import Vocabulary, CaptionDataset, Collate
from src.model import ImageCaptioningModel
from src.inference import greedy_predict

%matplotlib inline

## 1. Load Data
We load the train, dev, and test sets containing precomputed VGG19 image encodings (`encoding_with_vgg19`) and textual captions.

In [2]:
# Load precomputed VGG19 features
try:
    train_df = pd.read_parquet('../dataset/processed_dataset/train.parquet')
    dev_df = pd.read_parquet('../dataset/processed_dataset/dev.parquet')
    test_df = pd.read_parquet('../dataset/processed_dataset/test.parquet')
    print("Train:", train_df.shape)
    print("Dev:", dev_df.shape)
    print("Test:", test_df.shape)
except Exception as e:
    print(f"Failed to load parquet files: {e}\nPlease ensure you are running this notebook from the notebooks/ directory.")

Train: (32360, 4)
Dev: (4045, 4)
Test: (4050, 4)


## 2. Build Vocabulary
Fit the vocabulary on the training captions.

In [3]:
# Build vocabulary from training captions
vocab = Vocabulary(freq_threshold=2)

# Ensure captions start with appropriate tokenization format
captions_list = train_df['caption'].tolist()
vocab.build_vocabulary([c if c.startswith('<start>') else f"<start> {c} <end>" for c in captions_list])

print(f"Total vocabulary size: {len(vocab)}")

Total vocabulary size: 4956


## 3. Dataset and DataLoader
Create PyTorch dataset and dataloaders.

In [4]:
# Create PyTorch datasets
train_dataset = CaptionDataset(train_df, vocab)
dev_dataset = CaptionDataset(dev_df, vocab)
test_dataset = CaptionDataset(test_df, vocab)

pad_idx = vocab.stoi["<pad>"]
collate_fn = Collate(pad_idx=pad_idx)

# Hyperparameters
BATCH_SIZE = 128
NUM_WORKERS = 0

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(dataset=dev_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False, collate_fn=collate_fn)

## 4. Initialize Model
The model takes the 4096-dim VGG features, transforms them to the embedding space, and feeds them into an LSTM decoder.

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

EMBED_SIZE = 300
HIDDEN_SIZE = 256
VOCAB_SIZE = len(vocab)
NUM_LAYERS = 1
LEARNING_RATE = 3e-4
NUM_EPOCHS = 5

model = ImageCaptioningModel(embed_size=EMBED_SIZE, hidden_size=HIDDEN_SIZE, vocab_size=VOCAB_SIZE, num_layers=NUM_LAYERS)
model = model.to(device)

criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)

Using device: mps


ImageCaptioningModel(
  (encoder): Encoder(
    (linear): Linear(in_features=4096, out_features=300, bias=True)
    (relu): ReLU()
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (decoder): Decoder(
    (embed): Embedding(4956, 300)
    (lstm): LSTM(300, 256, batch_first=True)
    (linear): Linear(in_features=256, out_features=4956, bias=True)
    (dropout): Dropout(p=0.5, inplace=False)
  )
)


## 5. Training Loop

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/image_captioning

In [ ]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(log_dir="runs/image_captioning")
global_step = 0

In [6]:
import time

for epoch in range(NUM_EPOCHS):
    model.train()
    
    epoch_loss = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    
    for idx, (features, captions) in enumerate(progress_bar):
        features = features.to(device)
        captions = captions.to(device)
        
        # Forward pass
        outputs = model(features, captions)
        
        # Calculate loss (outputs shape: (N, seq_len, vocab_size), targets: (N, seq_len))
        loss = criterion(outputs.view(-1, VOCAB_SIZE), captions.view(-1))
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())
        
        # Add TensorBoard Logging
        writer.add_scalar('Loss/train', loss.item(), global_step)
        global_step += 1
        
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Average Loss: {epoch_loss/len(train_loader):.4f}")

writer.close()

Epoch 1/5:   0%|          | 0/253 [00:00<?, ?it/s]

Epoch [1/5] Average Loss: 4.7796


Epoch 2/5:   0%|          | 0/253 [00:00<?, ?it/s]

Epoch [2/5] Average Loss: 3.6806


Epoch 3/5:   0%|          | 0/253 [00:00<?, ?it/s]

Epoch [3/5] Average Loss: 3.4418


Epoch 4/5:   0%|          | 0/253 [00:00<?, ?it/s]

Epoch [4/5] Average Loss: 3.2947


Epoch 5/5:   0%|          | 0/253 [00:00<?, ?it/s]

Epoch [5/5] Average Loss: 3.1686


## 6. Evaluation / Inference
Let's pick a random image from the test set and generate a caption.

In [7]:
# Ensure model directory exists
os.makedirs('../weights', exist_ok=True)
torch.save(model.state_dict(), '../weights/caption_model.pth')
print("Model saved to weights/caption_model.pth")

Model saved to weights/caption_model.pth


In [8]:
def show_image_with_caption(index=None):
    if index is None:
        index = np.random.randint(0, len(test_df))
        
    row = test_df.iloc[index]
    image_path = os.path.join('..', 'dataset', 'Images', row['file'].split('/')[-1])
    features = torch.tensor(row['encoding_with_vgg19'], dtype=torch.float32).unsqueeze(0)
    actual_caption = row['caption']
    
    predicted_caption = greedy_predict(model, features, vocab, device=device)
    
    try:
        # Some paths in the original code pointed to a separate 'Images' directory that might not exist 
        # or were replaced by 'images'. We assume standard structure.
        if not os.path.exists(image_path):
            image_path = os.path.join('..', 'images', row['file'].split('/')[-1])
            
        if os.path.exists(image_path):
            img = Image.open(image_path)
            plt.imshow(img)
            plt.axis('off')
            plt.title(f"Predicted: {predicted_caption}")
            plt.show()
    except Exception as e:
        pass
        
    print(f"Image ID: {row['file']}")
    print(f"Actual: {actual_caption}")
    print(f"Predicted: {predicted_caption}")

show_image_with_caption()

Image ID: /kaggle/working/Image-Captioning/dataset/Images/127490019_7c5c08cb11.jpg
Actual: A woman wearing blue begins a golf swing .
Predicted: <start> A man in a red shirt is standing on a bench .
